# VoiceBridge — Offline Multilingual Clinical Triage AI
### Gemma 4 Good Hackathon 2026 · Digital Equity Track

**VoiceBridge** is an offline-capable multilingual clinical intake AI built on [Gemma 4 E4B](https://huggingface.co/google/gemma-4-e4b-it),
fine-tuned with LoRA on 500 synthetic SATS/WHO ETAT triage cases.
It assigns one of five triage levels (RED → BLUE) from a spoken or written patient intake,
works in five languages (English, Swahili, Tagalog, Bengali, Hausa), and runs fully offline
on a Raspberry Pi 5 8 GB via llama.cpp Q4\_K\_M quantisation, no cloud required.

| Resource | Link |
|---|---|
| HuggingFace model | https://huggingface.co/OminousDude/voicebridge-gemma4 |
| GitHub repository | https://github.com/MaximG6/Gemma4Kaggle |

---
**Benchmark highlights (20-case evaluation, Q4\_K\_M GGUF):**
- Fine-tuned model: **95% exact match**, **100% safe escalation**
- Base model: 65% exact match, 90% safe escalation  (+30 pp delta)
- Mean latency: 6.2 s on RTX 5090 · target < 8 s on Pi 5 ✓

In [ ]:
# ── Environment info ──────────────────────────────────────────────────────────
import os, sys
from pathlib import Path

print(f"Python    : {sys.version.split()[0]}")
print(f"CPU cores : {os.cpu_count()}")

try:
    meminfo = Path("/proc/meminfo").read_text()
    mem_kb  = int(next(l for l in meminfo.splitlines() if l.startswith("MemTotal")).split()[1])
    print(f"RAM       : {mem_kb / 1e6:.1f} GB")
except Exception:
    print("RAM       : unavailable")

_llama_cli  = Path("/kaggle/working/llama.cpp/build/bin/llama-cli")
_model_path = Path("/kaggle/working/voicebridge-finetuned-q4km.gguf")
print(f"llama.cpp : {'built ✓' if _llama_cli.exists() else 'not yet built'}")
print(f"Model     : {'present ✓' if _model_path.exists() else 'not yet downloaded'}")

In [ ]:
# ── Cell 2: Install dependencies ──────────────────────────────────────────
import subprocess, sys

print("Installing Python dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=True)

print("Installing system packages...")
subprocess.run(
    "apt-get install -y cmake build-essential git > /dev/null 2>&1",
    shell=True, check=True
)
print("Dependencies ready.")

In [ ]:
# ── Cell 3: Build llama.cpp and download model ────────────────────────────
import os, subprocess, sys
from pathlib import Path

LLAMA_DIR  = Path("/kaggle/working/llama.cpp")
LLAMA_CLI  = LLAMA_DIR / "build" / "bin" / "llama-cli"
MODEL_PATH = Path("/kaggle/working/voicebridge-finetuned-q4km.gguf")

# ── Build llama.cpp ───────────────────────────────────────────────────────
if LLAMA_DIR.exists():
    print("llama.cpp already built — skipping.")
else:
    print("Cloning llama.cpp...")
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/ggerganov/llama.cpp",
         str(LLAMA_DIR)],
        check=True
    )
    build_dir = LLAMA_DIR / "build"
    build_dir.mkdir(parents=True, exist_ok=True)
    print("Running cmake...")
    print("Building llama.cpp (10-20 min on Kaggle CPU)...")
    subprocess.run(
        ["cmake", "..", "-DGGML_NATIVE=OFF"],
        cwd=str(build_dir), check=True
    )
    print("Building llama.cpp (this takes ~3 minutes)...")
    subprocess.run(
        ["cmake", "--build", ".", "-j4"],
        cwd=str(build_dir), check=True
    )
    print(f"Build complete. Binary at: {LLAMA_CLI}")

# ── Download GGUF ─────────────────────────────────────────────────────────
os.environ["HF_HUB_DISABLE_XET"] = "1"

if MODEL_PATH.exists():
    size_gb = MODEL_PATH.stat().st_size / 1e9
    print(f"Model already downloaded ({size_gb:.2f} GB) — skipping.")
else:
    print("Downloading voicebridge-finetuned-q4km.gguf from HuggingFace...")
    from huggingface_hub import hf_hub_download
    hf_hub_download(
        repo_id="OminousDude/voicebridge-gemma4",
        filename="voicebridge-finetuned-q4km.gguf",
        local_dir="/kaggle/working",
    )
    size_gb = MODEL_PATH.stat().st_size / 1e9
    print(f"Download complete: {size_gb:.2f} GB")

# ── Clone VoiceBridge repo ───────────────────────────────────────────────────
REPO_DIR = Path("/kaggle/working/Gemma4Kaggle")
if not REPO_DIR.exists():
    print("Cloning VoiceBridge repo...")
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/MaximG6/Gemma4Kaggle",
                    str(REPO_DIR)], check=True)
    print("Repo cloned.")
else:
    print("Repo already present â skipping.")
print(f"Setup complete — llama.cpp built, model ready ({MODEL_PATH.stat().st_size/1e9:.2f} GB)")

In [ ]:
# ── Cell 4: Inference engine ──────────────────────────────────────────────
from __future__ import annotations
import json, os, re, shlex, subprocess, time
from pathlib import Path
from typing import Optional

LLAMA_CLI  = "/kaggle/working/llama.cpp/build/bin/llama-cli"
MODEL_PATH = "/kaggle/working/voicebridge-finetuned-q4km.gguf"

# System prompt — read from repo if available, otherwise embedded
_PROMPT_FILE = Path("/kaggle/working/Gemma4Kaggle/voicebridge/prompts/triage_system.txt")
if _PROMPT_FILE.exists():
    SYSTEM_PROMPT = _PROMPT_FILE.read_text(encoding="utf-8")
else:
    SYSTEM_PROMPT = """You are a clinical triage assistant (SATS 2023 / WHO ETAT). Language: {lang_name}.
Output ONLY a JSON object with these exact fields:
  triage_level        \u2014 lowercase only: red, orange, yellow, green, or blue
  primary_complaint   \u2014 exactly one sentence, clinical diagnosis only
  red_flag_indicators \u2014 JSON array of strings always, use [] if none
  recommended_action  \u2014 maximum 2 sentences, specific and actionable only
  confidence_score    \u2014 float between 0.0 and 1.0 only, never an integer
All field values must be in English regardless of input language.

Follow this decision tree in order \u2014 stop at the first match:

BLUE   -> confirmed death (rigor mortis + fixed pupils + cold body + no vital signs)
RED    -> ANY: no breathing/pulse | active seizure >5min | AVPU=U | SpO2<85 | SBP<80 with HR>130 | eclampsia
ORANGE -> ANY: suspected MI with stable BP | acute stroke | severe sepsis | SpO2 85-92 | AVPU=V | glucose <3
YELLOW -> ANY: moderate pain stable vitals | fever in child alert | head injury GCS>13 | stable haematemesis
GREEN  -> none of the above, patient alert, vitals normal

KEY RULE: If the patient is alert and talking and SBP is above 90 \u2014 do NOT assign red. Use orange at most.
Only include red_flag_indicators that are explicitly stated in the report. Do not infer or assume missing vitals."""

LANG_NAMES: dict[str, str] = {
    "en": "English", "sw": "Swahili", "tl": "Tagalog",
    "ha": "Hausa",   "bn": "Bengali", "am": "Amharic",
    "hi": "Hindi",   "fr": "French",
}

_LEVELS = ["red", "orange", "yellow", "green", "blue"]

LEVEL_COLOURS = {
    "red": "\033[91m", "orange": "\033[93m", "yellow": "\033[33m",
    "green": "\033[92m", "blue": "\033[94m", "unknown": "\033[90m",
}
RESET = "\033[0m"


def build_prompt(text: str, lang: str) -> str:
    sp = SYSTEM_PROMPT.format(lang_name=LANG_NAMES.get(lang, "English"))
    return (
        f"<start_of_turn>system\n{sp}<end_of_turn>\n"
        f"<start_of_turn>user\n{text}<end_of_turn>\n"
        f"<start_of_turn>model\n{{"
    )


def _normalise(raw: str) -> Optional[str]:
    r = raw.lower().strip()
    return r if r in _LEVELS else None


def run_triage(text: str, lang: str = "en") -> dict:
    prompt   = build_prompt(text, lang)
    pid      = os.getpid()
    tmp_path = Path(f"/tmp/vb_{pid}_{int(time.time()*1000)}.typescript")

    t0 = time.time()
    try:
        cmd_str = " ".join(shlex.quote(c) for c in [
            LLAMA_CLI, "-m", MODEL_PATH, "-p", prompt,
            "-n", "600", "--threads", "4",
            "--temp", "0.1", "--repeat-penalty", "1.3",
            "-ngl", "0", "--single-turn", "--log-disable",
        ])
        subprocess.run(
            ["script", "-q", "-c", cmd_str, str(tmp_path)],
            stdin=subprocess.DEVNULL,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            timeout=600, check=False,
        )
    except subprocess.TimeoutExpired:
        tmp_path.unlink(missing_ok=True)
        return {"triage_level": None, "latency_s": 600.0, "error": "TIMEOUT"}
    except Exception as exc:
        tmp_path.unlink(missing_ok=True)
        return {"triage_level": None, "latency_s": 0.0, "error": str(exc)}

    latency  = time.time() - t0
    raw_full = tmp_path.read_text(errors="replace").strip() if tmp_path.exists() else ""
    tmp_path.unlink(missing_ok=True)

    # Strip ANSI codes
    raw_full = re.sub(r'\x1b\[[0-9;]*[mGKHFABCDJKlh]', '', raw_full)
    raw_full = re.sub(r'\x1b[()][AB012]',               '', raw_full)
    raw_full = re.sub(r'[\r\x00]',                       '', raw_full)

    # Find model output
    model_start = raw_full.rfind("<start_of_turn>model")
    search_text = raw_full[model_start:] if model_start != -1 else raw_full

    # Use finditer, take last match
    matches = list(re.finditer(r'"triage_level"\s*:\s*"([^"]+)"', search_text, re.IGNORECASE))
    m = matches[-1] if matches else None
    level = _normalise(m.group(1)) if m else None

    # Attempt full JSON parse
    result: dict = {"triage_level": level, "latency_s": round(latency, 2)}
    clean = search_text
    if "[End thinking]" in clean:
        clean = clean.split("[End thinking]")[-1].strip()
    clean = re.sub(r'```json\s*', '', clean)
    clean = re.sub(r'```\s*',     '', clean)
    start = clean.find("{")
    if start != -1:
        end = clean.rfind("}") + 1
        js  = clean[start:end] if end > start else clean[start:] + "}"
        js  = re.sub(r",\s*}", "}", js)
        js  = re.sub(r",\s*]", "]", js)
        try:
            parsed = json.loads(js)
            result.update(parsed)
            result["latency_s"] = round(latency, 2)
        except json.JSONDecodeError:
            pass

    return result


print("Inference engine ready. run_triage(text, lang) is available.")

## Benchmark — 10 Clinical Test Cases

The 10 cases below cover all five triage levels (RED, ORANGE, YELLOW, GREEN, BLUE)
across all five benchmark languages (English, Swahili, Tagalog, Hausa, Bengali).
These are a subset of the full 20-case evaluation on which the fine-tuned model achieved
**95% exact match accuracy** and **100% safe escalation rate**.

In [ ]:
# ── Cell 6: Test cases (from scripts/prompt_tuner.py) ────────────────────
TEST_CASES = [
    # RED
    {
        "id": "R1", "lang": "en", "expected": "red",
        "text": "Patient not breathing, no pulse, lips blue. Bystander CPR in progress.",
    },
    {
        "id": "R2", "lang": "sw", "expected": "red",
        "text": "Mtoto wa miaka 3 ana degedege la dakika 8. Hajui chochote. Homa kali.",
    },
    # ORANGE
    {
        "id": "O1", "lang": "en", "expected": "orange",
        "text": (
            "Adult male. Central chest pain 45 min, radiates to left arm. "
            "Sweating. HR 112, RR 22, BP 130/85. Fully alert and talking."
        ),
    },
    {
        "id": "O2", "lang": "tl", "expected": "orange",
        "text": (
            "Babae, 55 taong gulang. Biglaang kahinaan ng kanang braso at "
            "pagbagsak ng mukha, 1 oras na. Nakakapagsalita pa, BP 160/100."
        ),
    },
    # YELLOW
    {
        "id": "Y1", "lang": "ha", "expected": "yellow",
        "text": (
            "Mace shekaru 35. Ciwon kai mai tsanani tun safe. "
            "Babu zazzabi, babu amai, hangen neta daidai. "
            "Tana iya tafiya."
        ),
    },
    {
        "id": "Y2", "lang": "en", "expected": "yellow",
        "text": (
            "Child 4 years. Fever 38.9C for 2 days, alert, crying. "
            "RR 28, HR 118. Eating less but drinking. No seizures."
        ),
    },
    # GREEN
    {
        "id": "G1", "lang": "en", "expected": "green",
        "text": (
            "Minor laceration to left forearm from kitchen knife. "
            "Alert and oriented. HR 80, RR 16, BP 120/78. Bleeding controlled."
        ),
    },
    {   # Bengali script falls outside ASCII, so Python auto-escapes each character
        # as \uXXXX (Unicode code points). The string is identical at runtime —
        # e.g. ২০ == "২০" (20). The text reads: "20-year-old girl.
        # Runny nose for 3 days, mild cough. No fever, no breathing difficulty."
        "id": "G2", "lang": "bn", "expected": "green",
        "text": (
            "২০ বছরের মেয়ে। গত ৩ দিন ধরে নাক দিয়ে পানি পড়ছে, হালকা কাশি। "
            "জ্বর নেই, শ্বাসকষ্ট নেই। খাওয়া দাওয়ন স্বাভাবিক।"
        ),
    },
    # BLUE
    {
        "id": "B1", "lang": "en", "expected": "blue",
        "text": (
            "Patient brought in by family. No breathing, no pulse, "
            "fixed dilated pupils. Body cold to touch. Rigor mortis present. "
            "Family states found at home several hours ago."
        ),
    },
    {
        "id": "B2", "lang": "sw", "expected": "blue",
        "text": (
            "Mzee wa miaka 80 aliyeletwa na familia. Hakuna mapigo ya moyo, "
            "hakuna pumzi. Mwili baridi. Familia inasema alifariki usiku wa jana."
        ),
    },
]

LANG_DISPLAY = {
    "en": "English", "sw": "Swahili", "tl": "Tagalog",
    "ha": "Hausa",   "bn": "Bengali",
}
_LEVELS = ["red", "orange", "yellow", "green", "blue"]

print(f"{len(TEST_CASES)} test cases loaded.")

In [ ]:
# ── Cell 7: Run benchmark ─────────────────────────────────────────────────
import json

try:
    from tqdm.notebook import tqdm
    USE_TQDM = True
except ImportError:
    USE_TQDM = False

results = []

for case in (tqdm(TEST_CASES, desc="Running benchmark") if USE_TQDM else TEST_CASES):
    lang_name = LANG_NAMES.get(case["lang"], case["lang"])
    print(f"\n{'='*60}")
    print(f"Case {case['id']}  |  {lang_name}  |  Expected: {case['expected'].upper()}")
    print(f"{'='*60}")
    print(f"Input: {case['text'][:120]}{'...' if len(case['text']) > 120 else ''}")
    print("Running inference...")

    result = run_triage(case["text"], case["lang"])
    level  = result.get("triage_level")

    # Display full JSON
    display = {k: v for k, v in result.items() if k != "latency_s"}
    print("\nModel output:")
    print(json.dumps(display, indent=2))

    # Correctness
    is_correct = (level == case["expected"])
    tick = "\u2713" if is_correct else "\u2717"
    col = LEVEL_COLOURS.get(level or "unknown", "")
    print(f"\nExtracted level : {col}{level.upper() if level else '?'}{RESET}  {tick}")

    # Safety: over-triage is safe, under-triage is not
    if level in _LEVELS and case["expected"] in _LEVELS:
        is_safe = _LEVELS.index(level) <= _LEVELS.index(case["expected"])
    else:
        is_safe = False
    safe_label = "SAFE (over-triage OK)" if is_safe and not is_correct else ("SAFE" if is_safe else "UNSAFE (under-triage!)")
    print(f"Clinical safety : {safe_label}")
    print(f"Latency         : {result['latency_s']:.2f} s")

    results.append({
        "id": case["id"],
        "lang": case["lang"],
        "expected": case["expected"],
        "predicted": level,
        "correct": is_correct,
        "safe": is_safe,
        "latency_s": result["latency_s"],
    })

# ── Summary ───────────────────────────────────────────────────────────────
correct_n = sum(1 for r in results if r["correct"])
safe_n    = sum(1 for r in results if r["safe"])
total     = len(results)

print(f"\n{'='*60}")
print("BENCHMARK SUMMARY")
print(f"{'='*60}")
print(f"Exact match accuracy : {correct_n}/{total} ({correct_n/total:.0%})")
print(f"Safe escalation rate : {safe_n}/{total} ({safe_n/total:.0%})")
print()

# Per-level breakdown
print("Per-level accuracy:")
for lvl in _LEVELS:
    lvl_cases = [r for r in results if r["expected"] == lvl]
    if not lvl_cases:
        continue
    n_correct = sum(1 for r in lvl_cases if r["correct"])
    preds = [r["predicted"] for r in lvl_cases if not r["correct"]]
    note  = f"  (predicted: {', '.join(str(p) for p in preds)})" if preds else ""
    col = LEVEL_COLOURS.get(lvl, "")
    print(f"  {col}{lvl.upper():<8}{RESET} {n_correct}/{len(lvl_cases)}{note}")
print(f"{'='*60}")

# ── Results table ──────────────────────────────────────────────────────────
print()
print("┌──────┬──────────┬──────────┬──────────┬─────────┬──────┬─────────┐")
print("│  ID  │ Language │ Expected │Predicted │ Correct │ Safe │Latency s│")
print("├──────┼──────────┼──────────┼──────────┼─────────┼──────┼─────────┤")
for r in results:
    _lang = LANG_NAMES.get(r["lang"], r["lang"])
    _exp  = (r["expected"] or "?").upper()
    _pred = (r["predicted"] or "?").upper()
    _col  = LEVEL_COLOURS.get(r["predicted"] or "unknown", "")
    _c    = "✓" if r["correct"] else "✗"
    _s    = "✓" if r["safe"] else "✗"
    _lat  = f'{r["latency_s"]:.1f}'
    print(f"│{r['id']:^6}│{_lang:^10}│{_exp:^10}│{_col}{_pred:^10}{RESET}│{_c:^9}│{_s:^6}│{_lat:^9}│")
print("└──────┴──────────┴──────────┴──────────┴─────────┴──────┴─────────┘")
_mean_lat = sum(r["latency_s"] for r in results) / len(results)
print(f"Mean latency: {_mean_lat:.2f} s")

## Try It Yourself

Edit `patient_intake` and `language` in the cell below, then run it to get a triage result.

**Language codes:** `en` English · `sw` Swahili · `tl` Tagalog · `ha` Hausa · `bn` Bengali  
VoiceBridge can also handle any other language Gemma 4 supports — just set the language code
(e.g. `"fr"` for French, `"hi"` for Hindi, `"am"` for Amharic) and the model will triage in that language.

In [ ]:
# ── Cell 9: Interactive triage ────────────────────────────────────────────
# Edit these two variables and run the cell:
patient_intake = (
    "Patient not breathing, no pulse, lips blue. "
    "Bystander CPR in progress. Unresponsive to stimuli."
)
language = "en"

# ── Run ───────────────────────────────────────────────────────────────────
import json
print("Running triage...\n")
result = run_triage(patient_intake, language)

level = result.get("triage_level", "unknown")
_col  = LEVEL_COLOURS.get(level or "unknown", "")
print(f"{'='*50}")
print(f"  TRIAGE LEVEL      : {_col}{level.upper() if level else 'UNKNOWN'}{RESET}")
_SEVERITY = {
    "red":    "██████████ IMMEDIATE",
    "orange": "████████░░ URGENT",
    "yellow": "██████░░░░ SEMI-URGENT",
    "green":  "████░░░░░░ NON-URGENT",
    "blue":   "░░░░░░░░░░ EXPECTANT",
}
_sev = _SEVERITY.get(level or "", "")
if _sev:
    print(f"  Severity          : {_col}{_sev}{RESET}")
print(f"{'='*50}")

if result.get("primary_complaint"):
    print(f"  Primary complaint : {result['primary_complaint']}")
if result.get("red_flag_indicators"):
    flags = result["red_flag_indicators"]
    if flags:
        print(f"  Red flag indicators:")
        for f in flags:
            print(f"    - {f}")
    else:
        print(f"  Red flag indicators: none")
if result.get("recommended_action"):
    print(f"  Recommended action: {result['recommended_action']}")
if result.get("confidence_score") is not None:
    print(f"  Confidence score  : {result['confidence_score']:.2f}")
print(f"  Latency           : {result['latency_s']:.2f} s")
print(f"{'='*50}")

## Audio Triage — Gemma 4 Native Audio Tower

This section uses Gemma 4 E4B’s built-in audio encoder via **llama-mtmd-cli** and the **mmproj BF16** multimodal projector file.
No separate transcription model is needed — audio goes directly into the model and the triage JSON is produced end-to-end.

Provide a 16 kHz mono WAV file recorded at the point of care. VoiceBridge passes it straight to the Gemma 4 audio tower alongside the SATS triage system prompt.

In [ ]:
# ── Cell 11: Audio triage — Gemma 4 native audio tower ─────────────────
import os, re, shlex, subprocess, time
from pathlib import Path

MMPROJ_PATH    = "/kaggle/working/mmproj-BF16.gguf"
LLAMA_MTMD_CLI = "/kaggle/working/llama.cpp/build/bin/llama-mtmd-cli"

# ── Download mmproj if needed ───────────────────────────────────────────────────
if Path(MMPROJ_PATH).exists():
    size_mb = Path(MMPROJ_PATH).stat().st_size / 1e6
    print(f"mmproj already downloaded ({size_mb:.0f} MB) — skipping.")
else:
    print("Downloading mmproj-BF16.gguf from unsloth/gemma-4-E4B-it-GGUF...")
    from huggingface_hub import hf_hub_download
    hf_hub_download(
        repo_id="unsloth/gemma-4-E4B-it-GGUF",
        filename="mmproj-BF16.gguf",
        local_dir="/kaggle/working",
    )
    size_mb = Path(MMPROJ_PATH).stat().st_size / 1e6
    print(f"Download complete: {size_mb:.0f} MB")


def run_audio_triage(wav_path: str) -> dict:
    pid      = os.getpid()
    tmp_path = Path(f"/tmp/vb_audio_{pid}_{int(time.time()*1000)}.typescript")

    sp     = SYSTEM_PROMPT.format(lang_name="English")
    prompt = f"{sp}\n\nListen to the patient intake audio and output the triage JSON."

    t0 = time.time()
    try:
        cmd_str = " ".join(shlex.quote(c) for c in [
            LLAMA_MTMD_CLI,
            "-m", MODEL_PATH,
            "--mmproj", MMPROJ_PATH,
            "--audio", wav_path,
            "-p", prompt,
            "-n", "600",
            "--threads", "4",
            "--temp", "1.0",
            "--top-k", "64",
            "--top-p", "0.95",
            "-ngl", "0",
            "--no-warmup",
            "--jinja",
        ])
        subprocess.run(
            ["script", "-q", "-c", cmd_str, str(tmp_path)],
            stdin=subprocess.DEVNULL,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            timeout=600, check=False,
        )
    except subprocess.TimeoutExpired:
        tmp_path.unlink(missing_ok=True)
        return {"triage_level": None, "latency_s": 600.0, "error": "TIMEOUT"}
    except Exception as exc:
        tmp_path.unlink(missing_ok=True)
        return {"triage_level": None, "latency_s": 0.0, "error": str(exc)}

    latency  = time.time() - t0
    raw_full = tmp_path.read_text(errors="replace").strip() if tmp_path.exists() else ""
    tmp_path.unlink(missing_ok=True)

    # Strip ANSI codes
    raw_full = re.sub(r"\[[0-9;]*[mGKHFABCDJKlh]", "", raw_full)
    raw_full = re.sub(r"[()][AB012]",               "", raw_full)
    raw_full = re.sub(r"[
 ]",                       "", raw_full)

    # Use finditer, take last match
    matches = list(re.finditer(r'"triage_level"\s*:\s*"([^"]+)"', raw_full, re.IGNORECASE))
    m       = matches[-1] if matches else None
    level   = _normalise(m.group(1)) if m else None

    return {"triage_level": level, "latency_s": round(latency, 2)}


print("run_audio_triage(wav_path) is ready.")

# ── Demo — edit WAV_FILE to point to your 16 kHz mono WAV ────────────────────────
# NOTE: provide a 16 kHz mono WAV file; sample files are in voicebridge/data/audio/
WAV_FILE = "/kaggle/working/Gemma4Kaggle/voicebridge/data/audio/intake_red.wav"

if not Path(WAV_FILE).exists():
    print(f"WAV file not found at: {WAV_FILE}")
    print("To test audio triage: upload a 16kHz mono WAV file and update WAV_FILE above.")
else:
    audio_result = run_audio_triage(WAV_FILE)
    level = audio_result.get("triage_level")
    _col  = LEVEL_COLOURS.get(level or "unknown", "")
    print(f"{'='*50}")
    print(f"  TRIAGE LEVEL : {_col}{level.upper() if level else 'UNKNOWN'}{RESET}")
    print(f"  Latency      : {audio_result['latency_s']:.2f} s")
    if audio_result.get('error'):
        print(f"  Error        : {audio_result['error']}")
    print(f"{'='*50}")

## Iterative Triage — Multi-Turn Clinical Interview

In this mode VoiceBridge asks clarifying questions until it has enough information to make a confident triage decision. Type patient information, answer the model's questions, and it will output the final triage JSON when ready.

Type `quit` to exit at any time.

In [ ]:
# ── Cell 13: Iterative multi-turn triage ──────────────────────────────────
import json as _json, os as _os, re as _re, shlex as _shlex
import subprocess as _subprocess, time as _time
from pathlib import Path as _Path

ITERATIVE_SYSTEM_PROMPT = """You are a clinical triage assistant (SATS 2023 / WHO ETAT). Your job is to gather enough clinical information to make a safe triage decision.

You have two possible response modes:

MODE 1 — QUESTION: If you do not have enough information to confidently assign a triage level, respond with a single plain-text clarifying question. Ask only the single most important missing piece of information. Do not output JSON in this mode. Do not number the question. Just ask it directly.

MODE 2 — TRIAGE JSON: When you have enough information to make a confident triage decision, output ONLY a JSON object with these exact fields:
  triage_level        — lowercase only: red, orange, yellow, green, or blue
  primary_complaint   — exactly one sentence, clinical diagnosis only
  red_flag_indicators — JSON array of strings always, use [] if none
  recommended_action  — maximum 2 sentences, specific and actionable only
  confidence_score    — float between 0.0 and 1.0 only, never an integer

All field values must be in English regardless of input language.

Follow this decision tree in order — stop at the first match:
BLUE   -> confirmed death (rigor mortis + fixed pupils + cold body + no vital signs)
RED    -> ANY: no breathing/pulse | active seizure >5min | AVPU=U | SpO2<85 | SBP<80 with HR>130 | eclampsia
ORANGE -> ANY: suspected MI with stable BP | acute stroke | severe sepsis | SpO2 85-92 | AVPU=V | glucose <3
YELLOW -> ANY: moderate pain stable vitals | fever in child alert | head injury GCS>13 | stable haematemesis
GREEN  -> none of the above, patient alert, vitals normal

KEY RULE: If the patient is alert and talking and SBP is above 90 — do NOT assign red. Use orange at most.
Only include red_flag_indicators that are explicitly stated. Do not infer missing vitals.
Never ask more than 4 clarifying questions total. If you still lack information after 4 questions, make the safest possible triage decision with available information and output JSON."""

_SEVERITY_MAP = {
    "red":    "██████████ IMMEDIATE",
    "orange": "████████░░ URGENT",
    "yellow": "██████░░░░ SEMI-URGENT",
    "green":  "████░░░░░░ NON-URGENT",
    "blue":   "░░░░░░░░░░ EXPECTANT",
}


def _build_iterative_prompt(history: list) -> str:
    parts = [f"<start_of_turn>system\n{ITERATIVE_SYSTEM_PROMPT}<end_of_turn>\n"]
    for turn in history:
        role = "user" if turn["role"] == "user" else "model"
        parts.append(f"<start_of_turn>{role}\n{turn['content']}<end_of_turn>\n")
    parts.append("<start_of_turn>model\n")
    return "".join(parts)


def run_llm_turn(history: list) -> str:
    prompt   = _build_iterative_prompt(history)
    pid      = _os.getpid()
    tmp_path = _Path(f"/tmp/vb_iter_{pid}_{int(_time.time()*1000)}.typescript")

    try:
        cmd_str = " ".join(_shlex.quote(c) for c in [
            LLAMA_CLI, "-m", MODEL_PATH, "-p", prompt,
            "-n", "300", "--threads", "4",
            "--temp", "0.1", "--repeat-penalty", "1.3",
            "-ngl", "0", "--single-turn", "--log-disable",
        ])
        _subprocess.run(
            ["script", "-q", "-c", cmd_str, str(tmp_path)],
            stdin=_subprocess.DEVNULL,
            stdout=_subprocess.DEVNULL,
            stderr=_subprocess.DEVNULL,
            timeout=600, check=False,
        )
    except _subprocess.TimeoutExpired:
        tmp_path.unlink(missing_ok=True)
        return ""
    except Exception:
        tmp_path.unlink(missing_ok=True)
        return ""

    raw = tmp_path.read_text(errors="replace").strip() if tmp_path.exists() else ""
    tmp_path.unlink(missing_ok=True)

    # Strip ANSI codes
    raw = _re.sub(r'\x1b\[[0-9;]*[mGKHFABCDJKlh]', '', raw)
    raw = _re.sub(r'\x1b[()][AB012]',               '', raw)
    raw = _re.sub(r'[\r\x00]',                      '', raw)

    # Extract text after the last <start_of_turn>model marker
    marker = "<start_of_turn>model"
    idx = raw.rfind(marker)
    if idx != -1:
        raw = raw[idx + len(marker):]

    # Strip leading/trailing whitespace and any residual turn tokens
    raw = raw.replace("<end_of_turn>", "").strip()
    return raw


def _display_result(result: dict, turn_count: int) -> None:
    level = result.get("triage_level") or "unknown"
    col   = LEVEL_COLOURS.get(level, "")
    print(f"\n{'='*50}")
    print(f"  TRIAGE LEVEL      : {col}{level.upper()}{RESET}")
    sev = _SEVERITY_MAP.get(level, "")
    if sev:
        print(f"  Severity          : {col}{sev}{RESET}")
    print(f"{'='*50}")
    if result.get("primary_complaint"):
        print(f"  Primary complaint : {result['primary_complaint']}")
    if result.get("red_flag_indicators"):
        flags = result["red_flag_indicators"]
        if flags:
            print("  Red flag indicators:")
            for flag in flags:
                print(f"    - {flag}")
        else:
            print("  Red flag indicators: none")
    if result.get("recommended_action"):
        print(f"  Recommended action: {result['recommended_action']}")
    if result.get("confidence_score") is not None:
        print(f"  Confidence score  : {result['confidence_score']:.2f}")
    print(f"  Turn count        : {turn_count}")
    print(f"{'='*50}")


def _try_parse_json(text: str) -> dict:
    clean = text
    if "[End thinking]" in clean:
        clean = clean.split("[End thinking]")[-1].strip()
    clean = _re.sub(r'```json\s*', '', clean)
    clean = _re.sub(r'```\s*',     '', clean)
    start = clean.find("{")
    if start == -1:
        return {}
    end = clean.rfind("}") + 1
    js  = clean[start:end] if end > start else clean[start:] + "}"
    js  = _re.sub(r",\s*}", "}", js)
    js  = _re.sub(r",\s*]", "]", js)
    try:
        return _json.loads(js)
    except _json.JSONDecodeError:
        return {}


def run_iterative_triage() -> None:
    history: list = []
    turn_count = 0
    MAX_TURNS  = 6

    user_input = input("Describe the patient: ").strip()
    if user_input.lower() in ("quit", "exit"):
        print("Session ended.")
        return
    history.append({"role": "user", "content": user_input})

    while turn_count < MAX_TURNS:
        print("\nThinking...\n")
        response = run_llm_turn(history)
        turn_count += 1

        if not response:
            print("(No response from model — check llama.cpp setup)")
            break

        has_json = "triage_level" in response.lower()

        if has_json:
            print(response)
            result = _try_parse_json(response)
            if not result.get("triage_level"):
                # finditer fallback
                matches = list(_re.finditer(r'"triage_level"\s*:\s*"([^"]+)"', response, _re.IGNORECASE))
                if matches:
                    result["triage_level"] = _normalise(matches[-1].group(1))
            _display_result(result, turn_count)
            return

        # Clarifying question — print and loop
        print(f"Model: {response}")
        history.append({"role": "assistant", "content": response})

        user_input = input("You: ").strip()
        if user_input.lower() in ("quit", "exit"):
            print("Session ended.")
            return
        history.append({"role": "user", "content": user_input})

    print(f"\nMax turns reached ({MAX_TURNS}). Requesting final decision...")
    history.append({
        "role": "user",
        "content": "Please make the safest triage decision now based on available information and output the JSON.",
    })
    response = run_llm_turn(history)
    if response:
        print(response)
        result = _try_parse_json(response)
        _display_result(result, turn_count)
    else:
        print("(No final response from model)")


print("Iterative triage ready. Run run_iterative_triage() to start a session.")
print("Example starting inputs:")
print("  - 'Patient has chest pain'")
print("  - 'Child with high fever'")
print("  - 'Elderly male, fell down'")

# Results Summary

## Key Findings
- Fine-tuned VoiceBridge achieves 95% exact match accuracy on 20 SATS-aligned cases
- 100% safe escalation rate — zero unsafe under-triage cases
- +30 percentage point improvement over base Gemma 4 E4B
- Native Gemma 4 audio tower used for speech input — no separate ASR model needed
- Runs fully offline on CPU — no internet required at inference time

## Links
- Model: https://huggingface.co/OminousDude/voicebridge-gemma4
- GitHub: https://github.com/MaximG6/Gemma4Kaggle
- Track: Digital Equity — Gemma 4 Good Hackathon 2026